# Using Local LLM With LangChain

In [1]:
  !pip install langchain langchain-ollama langchain-community

   ---------------------------------------- 0.0/570.0 kB ? eta -:--:--
   ---------------------------------------- 570.0/570.0 kB 8.1 MB/s  0:00:00
   ---------------------------------------- 0.0/744.6 kB ? eta -:--:--
   ---------------------------------------- 744.6/744.6 kB 7.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB 4.3 MB/s eta 0:00:01
   -------------------------- ------------- 1.6/2.4 MB 4.3 MB/s eta 0:00:01
   ----------------------------------- ---- 2.1/2.4 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 3.5 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? e


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Check if Ollama is Responding

In [8]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="llama3.2",
    temperature=0.5,
    num_predict=512,
    top_k=10,
    top_p=0.9,
    repeat_penalty=1.2,
)
response = llm.invoke("What is the $ value of USA in 2022 compare to indian inr?")
print(response.content)


The exchange rate between the US Dollar (USD) and Indian Rupee (INR) fluctuates constantly due to market forces. However, I can provide you with some approximate values based on historical data.

As of December 31, 2022:

1 USD was approximately equal to:
7-8 INR

So, if we use the average exchange rate for 2022, which is around 7.5 INR/USD, here are some examples:

* 100 USD = Approximately 750 INR
* 500 USD = Approximately 3750 INR
* 1,000 USD = Approximately 7500 INR

Please note that these values may not reflect the current exchange rate (as I'm a large language model, my knowledge cutoff is December 2023). The actual value of 1 USD in relation to INR might be different at this time.


### Testing with local LLM as actual_output source and evaluator

In [11]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=r"D:\Udemy_AI_project\Session1_Intro\.env")

from deepeval.models import OllamaModel
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
from deepeval.evaluate import evaluate
from deepeval.evaluate.configs import AsyncConfig

# Use Ollama as both the answer source (actual_output) and the evaluator model.
# No OpenAI API key is needed for evaluation.

confident_key = os.getenv("CONFIDENT_API_KEY")
if not confident_key:
    raise RuntimeError("CONFIDENT_API_KEY is missing. Set it in the environment or .env file.")

llm_model = OllamaModel(model="llama3.2:latest", base_url="http://localhost:11434")

# GEval compares actual vs expected output using the evaluator LLM.
# In DeepEval 4.x, GEval requires evaluation_params: the test-case fields it should read.
correctness_metric = GEval(
    name="Correctness",
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
    criteria="The actual output must state the same answer as the expected output. The actual output is correct if it identifies the same person or fact, even if it uses more words.",
    evaluation_steps=[
        "Read the input question.",
        "Read the expected output and the actual output.",
        "Check whether the actual output states the same core answer as the expected output.",
        "If the core answer matches, mark the test as passing; otherwise mark it as failing.",
    ],
    model=llm_model,
    threshold=0.5,
    async_mode=True,
    strict_mode=False,
)

test_case1 = LLMTestCase(
    input="Who is the president of United States Of America in 2022?",
    actual_output=llm.invoke("Who is the president of United States Of America in 2022?").content,
    expected_output="Joe Biden",
    retrieval_context=["The president of the United States is Joe Biden in 2022."],
)

test_case2 = LLMTestCase(
    input="Who built the Taj Mahal?",
    actual_output=llm.invoke("Who built the Taj Mahal?").content,
    expected_output="Shah Jahan",
    retrieval_context=["The Taj Mahal was built by Mughal Emperor Shah Jahan."],
)

print("Evaluating test cases...")
evaluate(test_cases=[test_case1, test_case2], metrics=[correctness_metric], async_config=AsyncConfig(run_async=False))


Evaluating test cases...


✨ You're running DeepEval's latest Correctness [GEval] Metric! (using llama3.2:latest (Ollama), strict=False, 
async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                        ┃ Average Score        ┃ Pass Rate                                   ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Correctness [GEval]           │ 0.70                 │ 100.00% | passed=2 | failed=0               │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=3142926;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=3142929;https://app.confident-ai.com/project/cmsvjmkke0001nz0vr8ajlqjx/test-runs/cmt3atr520002l50vvbjwk5zr/test-cases\https://app.confident-ai.com/project/cmsvjmkke0001nz0vr8ajlqjx/test-runs/cmt3atr520002l50vvbjwk5zr/test-cases]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=True, score=0.6, reason="The actual output provides the full name of the president, including the year of their term, whereas the expected output only requires the president's name. This discrepancy results in a score of 6, indicating partial alignment with the evaluation steps.", strict_mode=False, flaky=False, evaluation_model='llama3.2:latest (Ollama)', error=None, evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs='Criteria:\nThe actual output must state the same answer as the expected output. The actual output is correct if it identifies the same person or fact, even if it uses more words. \n \nEvaluation Steps:\n[\n    "Read the input question.",\n    "Read the expected output and the actual output.",\n    "Check whether the actual output states the same core answer as the expected output.",\n    "If the core answe